## Test Simple Sampling - sampler

In [ ]:
%load_ext autoreload
%autoreload 1

from cdxcore.verbose import Context
verbose = Context("all")
with verbose.write_t("Importing torch...", end='') as tme:   
    import torch
    verbose.write(f"done; this took {tme}.", head=False)

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.timeseries
%aimport quantdata.torch
%aimport quantdata.torch.ffw
%aimport quantdata.dysanos
%aimport quantdata.dysanos.simple
%aimport quantdata.dysanos.simple.sampler

from quantdata.widb.timeseries import DateRange, build_time_series, TimeSeriesDefinition, PoolConfig
from quantdata.widb.options_filter import ExpiryFilter, RawDayFilter, MkArbFree, OptionsFilter
from quantdata.dysanos.simple.sampler import SimpleSampler, SimpleConfig, SamplingConfig
from cdxcore.verbose import Context
from datetime import date

def test():
    simple_config = SimpleConfig(
            tickers = ['SPX'],
            date_range = DateRange(start=date(2025, 4, 1), end=date(2025, 7, 1)),
            raw_day_filter = RawDayFilter(),
            expiry_filter = ExpiryFilter(),
            options_filter = OptionsFilter(),
            mk_arb_free = MkArbFree("surface", smoothness=0.0),
            sampling_config = SamplingConfig(num_steps=10)
    )
    pool_config = PoolConfig(num_workers=4, threading=False) # no parallel processing for testing
    verbose = Context.all

    sampler = SimpleSampler(simple_config=simple_config, pool_config=pool_config, cache_mode="on", verbose=verbose)
    return sampler

sampler = test()
print("/done")

## Test shapes

In [ ]:
"""
Shapes
"""

from prettytable import PrettyTable

def test_shapes():
    batch_size = 7
    shapes = sampler.shapes( batch_size )

    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Shape"]

    for k, v in shapes.items():
        tbl.add_row([k, v])

    print(f"Shapes of batch size {batch_size}:")
    print(tbl)

test_shapes()   



## Test Batching

In [ ]:
from prettytable import PrettyTable
from cdxcore.util import fmt_list, fmt_big_byte_number

def test_batch():
    print("Loading torch...", end='', flush=True)
    import torch as torch
    print("done.")
    g = torch.Generator(device="cpu")
    g.manual_seed(1234)

    batch_size = 7
    print(f"\nCreating a batch of size {batch_size}:")

    batch_ix, batch = sampler(g, trn=True, batch_size=batch_size, device="cpu", return_batch_indices=True)

    # indices
    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Date"]

    for bix in batch_ix:
        date = sampler.data[bix.ticker].dates[bix.index]
        tbl.add_row([bix.ticker, bix.date])
    print(tbl)

    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Shape", "Type", "Size", "Device"]

    for k, v in batch.items():
        tbl.add_row([k, tuple(v.shape), v.dtype, fmt_big_byte_number(v.nbytes), v.device])
    
    print(tbl)

test_batch()



## Show Implied Vols

In [ ]:
from prettytable import PrettyTable
from cdxcore.util import fmt_list, fmt_big_byte_number
from cdxcore.dynaplot import figure

def viz_batch():
    print("Loading torch...", end='', flush=True)
    import torch as torch
    import numpy as nnp
    print("done.")
    g = torch.Generator(device="cpu")
    g.manual_seed(1234)
    batch_size = 7
    batch_ids, batch = sampler(g, trn=True, batch_size=batch_size, device="cpu", return_batch_indices=True)

    def viz(ibatch,istep):

        bis         = batch_ids[ibatch]
        is_valid_it = batch['is_valid'][ibatch,istep,:].detach().numpy()
        ask_vol_it  = batch['ask_vol'][ibatch,istep,:].detach().numpy()[is_valid_it]
        bid_vol_it  = batch['bid_vol'][ibatch,istep,:].detach().numpy()[is_valid_it]
        fit_vol_it  = batch['fit_vol'][ibatch,istep,:].detach().numpy()[is_valid_it]
        fitX_vol_it = batch['fitX_vol'][ibatch,istep,:].detach().numpy()[is_valid_it]
        afmask_t    = batch['mask_arb_free'][ibatch,istep,:].detach().numpy()[is_valid_it]
        nms_it      = batch['nms'][ibatch,istep,:].detach().numpy()[is_valid_it]
        tte_y       = batch['tte_y'][ibatch,istep,:].detach().numpy()[is_valid_it]

        unique_tte  = nnp.unique( tte_y )
        first = False
        with figure(f"Batch {bis.ticker} {bis.date}, Batch {ibatch} Step {istep}", row_size=3, col_size=3, columns=6) as fig:
            for tt in unique_tte:
                ax = fig.add_subplot(f"{tt*252:.2f}d")
                
                ask_vol_it_tt  = ask_vol_it[tte_y == tt]
                bid_vol_it_tt  = bid_vol_it[tte_y == tt]
                fit_vol_it_tt  = fit_vol_it[tte_y == tt]
                fitX_vol_it_tt = fitX_vol_it[tte_y == tt]
                afmask_t_tt    = afmask_t[tte_y == tt]
                nms_it_tt      = nms_it[tte_y == tt]
                ixs = nnp.argsort( nms_it_tt )

                ask_vol_it_tt = ask_vol_it_tt[ixs]
                bid_vol_it_tt = bid_vol_it_tt[ixs]
                fit_vol_it_tt = fit_vol_it_tt[ixs]
                fitX_vol_it_tt = fitX_vol_it_tt[ixs]
                afmask_t_tt    = afmask_t_tt[ixs]
                nms_it_tt     = nms_it_tt[ixs]

                ax.plot(nms_it_tt, ask_vol_it_tt, "v-", label="Ask Vol", color="red")
                ax.plot(nms_it_tt, bid_vol_it_tt, "^-", label="Bid Vol", color="green")
                ax.plot(nms_it_tt, fitX_vol_it_tt, "*-", label="FitX Vol", color="pink")
                ax.plot(nms_it_tt, fit_vol_it_tt, ".-", label="Fit Vol", color="orange")
                ax.plot(nms_it_tt[afmask_t_tt], fit_vol_it_tt[afmask_t_tt], "*", color="blue")

                if first:
                    ax.legend()
                    first = False
    
    for ibatch in range(batch_size):
        for istep in range(batch['is_valid'].shape[1]):
            viz(ibatch,istep)

viz_batch()



In [ ]:
print(sampler.data['SPX'].dates)

In [ ]:
assert set(sampler.data['SPX']) == sampler.config.features()|{'dates'}